# Visual Agents (UI Navigation) w/ Qwen2-VL

## Task Description
Simulate a Visual Agent that can look at a user interface (UI) and locate specific elements (buttons, fields) by coordinates. This is the foundation of "Computer Use" agents.

**Focus:** We use `Qwen/Qwen2-VL-2B-Instruct`, a vision-language model capable of grounding (finding objects).

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## Learning Objectives
(Will be updated)

In [ ]:
# Check for GPU
!nvidia-smi

## 1. Installation
We need `transformers`, `qwen-vl-utils`.

In [ ]:
!pip install -q transformers qwen-vl-utils accelerate torchvision

## 2. Imports

In [ ]:
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

## 3. Load Data
We create a simple synthetic UI image containing a 'Search' box and a 'Submit' button.

In [ ]:

# Create a dummy image
width, height = 640, 480
image = Image.new('RGB', (width, height), color='white')
draw = ImageDraw.Draw(image)

# Draw "Search" Box
draw.rectangle([50, 50, 400, 100], outline='black', width=2)
draw.text((60, 60), "Search query...", fill='gray')

# Draw "Submit" Button
draw.rectangle([420, 50, 550, 100], fill='blue', outline='black')
draw.text((450, 65), "Submit", fill='white')

# Draw "Cancel" Button
draw.rectangle([420, 120, 550, 170], fill='red', outline='black')
draw.text((450, 135), "Cancel", fill='white')

try:
    display(image)
except:
    pass
print("Created synthetic UI image.")


## 4. Data Cleaning
We define the user prompt to find the 'Submit' button.

In [ ]:
prompt = "Find the bounding box of the 'Submit' button."

## 5. Load Model
Load Qwen2-VL-2B-Instruct.

In [ ]:

model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print("Model loaded.")


## 6. Inference
Ask the model to locate the element.

In [ ]:

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Output: {output_text}")

# Parse the bounding box (Qwen-VL returns raw text like <box2d>...)
# Usually formatted as <box_2d>[y1, x1, y2, x2]</box_2d> or similar, normalized 0-1000.
# But specific handling might be needed. Qwen2-VL usually outputs: <|box_start|>(y1,x1),(y2,x2)<|box_end|>
# Note: The output format depends on the prompt. For 'Find the bounding box', it often returns the standard detection string.
# Let's save the result text.
results = output_text


## 7. Save Results
We also try to draw the box if we can parse it, otherwise we save the text.

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print("Try asking for 'Cancel' button or 'Search' box.")

## 9. Troubleshooting